In [1]:
!pip install gradio mtcnn lz4 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 43.2 MB/s eta 0:00:00


In [2]:
# Step 1 - Uninstall conflicting versions
import subprocess
subprocess.run(["pip", "uninstall", "mtcnn", "joblib", "lz4", "-y"], capture_output=True)

# Step 2 - Install compatible versions together
subprocess.run(["pip", "install", "lz4==4.0.0", "joblib==1.2.0", "mtcnn==0.1.1", "-q"])

print("Done ✅")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.7/163.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 298.0/298.0 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 43.7 MB/s eta 0:00:00
Done ✅


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tobler 0.13.0 requires joblib>=1.4, but you have joblib 1.2.0 which is incompatible.


In [3]:
!pip install gradio -q

import gradio as gr
import numpy as np
import cv2
import math
import os
import urllib.request
import warnings
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from PIL import Image
from scipy.ndimage import gaussian_filter

# ══════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════
DROP_PATH_RATE  = 0.1
DROPOUT         = 0.1
IMG_SIZE        = (128, 128)
THRESHOLD       = 0.5
MODEL_PATH      = "/kaggle/input/datasets/sahanakaids/cswin-checkpoints-new/cswin_best.keras"
OUT_DIR         = "/kaggle/working"
SHAP_N_BG       = 30
OCC_PATCH_SIZE  = 8

# ══════════════════════════════════════════════════════════
# ARCHITECTURE
# ══════════════════════════════════════════════════════════
class WarmupCosineDecay(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self,peak_lr,min_lr,warmup_steps,total_steps,**kwargs):
        super().__init__(**kwargs)
        self.peak_lr=float(peak_lr);self.min_lr=float(min_lr)
        self.warmup_steps=float(warmup_steps);self.total_steps=float(total_steps)
    def __call__(self,step):
        step=tf.cast(step,tf.float32)
        warmup=self.peak_lr*(step/tf.maximum(self.warmup_steps,1.0))
        cos_arg=math.pi*(step-self.warmup_steps)/tf.maximum(self.total_steps-self.warmup_steps,1.0)
        cosine=self.min_lr+0.5*(self.peak_lr-self.min_lr)*(1.0+tf.cos(cos_arg))
        return tf.where(step<self.warmup_steps,warmup,cosine)
    def get_config(self):
        return {"peak_lr":self.peak_lr,"min_lr":self.min_lr,
                "warmup_steps":self.warmup_steps,"total_steps":self.total_steps}

class DropPath(layers.Layer):
    def __init__(self,drop_prob=0.0,**kwargs):
        super().__init__(**kwargs);self.drop_prob=float(drop_prob)
    def call(self,x,training=False):
        if not training or self.drop_prob==0.0: return x
        keep=1.0-self.drop_prob
        shape=(tf.shape(x)[0],)+(1,)*(len(x.shape)-1)
        return x*tf.math.floor(keep+tf.random.uniform(shape,dtype=x.dtype))/keep
    def get_config(self):
        cfg=super().get_config();cfg["drop_prob"]=self.drop_prob;return cfg

class PatchEmbedding(layers.Layer):
    def __init__(self,dim,**kwargs):
        super().__init__(**kwargs);self.dim=dim
        self.proj=layers.Conv2D(dim,kernel_size=4,strides=4,padding="same",name="patch_proj")
        self.norm=layers.LayerNormalization(epsilon=1e-5,name="patch_norm")
    def build(self,input_shape):
        H=int(input_shape[1]);W=int(input_shape[2])
        self.pos_embed=self.add_weight(
            name="pos_embed",shape=(1,H//4,W//4,self.dim),
            initializer="zeros",trainable=True,dtype=tf.float32)
        super().build(input_shape)
    def call(self,x):
        return self.norm(self.proj(x)+tf.cast(self.pos_embed,x.dtype))
    def get_config(self):
        cfg=super().get_config();cfg["dim"]=self.dim;return cfg

def h_split(x,s):
    B=tf.shape(x)[0];H=tf.shape(x)[1];W=tf.shape(x)[2];C=x.shape[-1]
    return tf.reshape(tf.transpose(tf.reshape(x,[B,H//s,s,W,C]),[0,1,3,2,4]),[B*(H//s),W*s,C])
def h_merge(x,B,H,W,C,s):
    return tf.reshape(tf.transpose(tf.reshape(x,[B,H//s,W,s,C]),[0,1,3,2,4]),[B,H,W,C])
def v_split(x,s):
    B=tf.shape(x)[0];H=tf.shape(x)[1];W=tf.shape(x)[2];C=x.shape[-1]
    return tf.reshape(tf.transpose(tf.reshape(x,[B,H,W//s,s,C]),[0,2,1,3,4]),[B*(W//s),H*s,C])
def v_merge(x,B,H,W,C,s):
    return tf.reshape(tf.transpose(tf.reshape(x,[B,W//s,H,s,C]),[0,2,1,3,4]),[B,H,W,C])

class StripeAttention(layers.Layer):
    def __init__(self,dim,num_heads,attn_drop=0.0,**kwargs):
        super().__init__(**kwargs)
        self.num_heads=num_heads;self.head_dim=dim//num_heads;self.scale=self.head_dim**-0.5
        self.qkv=layers.Dense(dim*3,use_bias=True,name="qkv")
        self.proj=layers.Dense(dim,use_bias=True,name="proj")
        self.drop=layers.Dropout(attn_drop)
    def call(self,x,training=False):
        B=tf.shape(x)[0];N=tf.shape(x)[1];C=x.shape[-1]
        qkv=tf.transpose(tf.reshape(self.qkv(x),[B,N,3,self.num_heads,self.head_dim]),[2,0,3,1,4])
        q,k,v=qkv[0],qkv[1],qkv[2]
        attn=self.drop(tf.nn.softmax(tf.matmul(q,k,transpose_b=True)*self.scale,axis=-1),training=training)
        return self.proj(tf.reshape(tf.transpose(tf.matmul(attn,v),[0,2,1,3]),[B,N,C]))
    def get_config(self):
        cfg=super().get_config();cfg.update({"dim":self.num_heads*self.head_dim,"num_heads":self.num_heads});return cfg

class CSWinAttention(layers.Layer):
    def __init__(self,dim,num_heads,split_size,attn_drop=0.0,**kwargs):
        super().__init__(**kwargs)
        assert dim%2==0;self.split_size=split_size;self.dim_half=dim//2
        hh=max(1,num_heads//2)
        self.attn_h=StripeAttention(self.dim_half,hh,attn_drop,name="attn_h")
        self.attn_v=StripeAttention(self.dim_half,hh,attn_drop,name="attn_v")
        self.lepe_h=layers.DepthwiseConv2D(3,padding="same",name="lepe_h")
        self.lepe_v=layers.DepthwiseConv2D(3,padding="same",name="lepe_v")
    def call(self,x,training=False):
        B=tf.shape(x)[0];H=tf.shape(x)[1];W=tf.shape(x)[2];s=self.split_size
        x1,x2=tf.split(x,2,axis=-1)
        xh=h_merge(self.attn_h(h_split(x1,s),training=training),B,H,W,self.dim_half,s)+self.lepe_h(x1)
        xv=v_merge(self.attn_v(v_split(x2,s),training=training),B,H,W,self.dim_half,s)+self.lepe_v(x2)
        return tf.concat([xh,xv],axis=-1)
    def get_config(self):
        cfg=super().get_config();cfg.update({"dim":self.dim_half*2,"num_heads":self.attn_h.num_heads*2,"split_size":self.split_size});return cfg

class CSWinBlock(layers.Layer):
    def __init__(self,dim,num_heads,split_size,mlp_ratio=4.0,drop_path=0.0,proj_drop=0.0,**kwargs):
        super().__init__(**kwargs)
        self.norm1=layers.LayerNormalization(epsilon=1e-5,name="norm1")
        self.attn=CSWinAttention(dim,num_heads,split_size,name="attn")
        self.dp1=DropPath(drop_path,name="dp1")
        self.norm2=layers.LayerNormalization(epsilon=1e-5,name="norm2")
        self.mlp=keras.Sequential([
            layers.Dense(int(dim*mlp_ratio),activation="gelu",name="fc1"),
            layers.Dropout(proj_drop),
            layers.Dense(dim,name="fc2"),
            layers.Dropout(proj_drop)],name="mlp")
        self.dp2=DropPath(drop_path,name="dp2")
    def call(self,x,training=False):
        x=x+self.dp1(self.attn(self.norm1(x),training=training),training=training)
        x=x+self.dp2(self.mlp(self.norm2(x),training=training),training=training)
        return x
    def get_config(self):
        cfg=super().get_config();cfg.update({"dim":self.attn.dim_half*2,"num_heads":self.attn.attn_h.num_heads*2,"split_size":self.attn.split_size});return cfg

class PatchMerging(layers.Layer):
    def __init__(self,out_dim,**kwargs):
        super().__init__(**kwargs);self.out_dim=out_dim
        self.conv=layers.Conv2D(out_dim,kernel_size=2,strides=2,padding="same",name="down_conv")
        self.norm=layers.LayerNormalization(epsilon=1e-5,name="down_norm")
    def call(self,x): return self.norm(self.conv(x))
    def get_config(self):
        cfg=super().get_config();cfg["out_dim"]=self.out_dim;return cfg

class CSWinTransformer(keras.Model):
    def __init__(self,embed_dim=64,depths=(2,2,6,2),num_heads=(2,4,8,16),
                 split_sizes=(2,2,4,2),mlp_ratio=4.0,drop_path_rate=DROP_PATH_RATE,
                 proj_drop=DROPOUT,num_classes=1,**kwargs):
        super().__init__(**kwargs)
        dpr=list(np.linspace(0,drop_path_rate,sum(depths)));bi=0;dim=embed_dim
        self.patch_embed=PatchEmbedding(embed_dim,name="patch_embed");self._sc=[]
        for si,(d,h,s) in enumerate(zip(depths,num_heads,split_sizes)):
            ns=[]
            for b in range(d):
                n=f"s{si}_b{b}";setattr(self,n,CSWinBlock(dim,h,s,mlp_ratio,dpr[bi],proj_drop,name=n));ns.append(n);bi+=1
            dn=None
            if si<len(depths)-1:
                dn=f"down_{si}";setattr(self,dn,PatchMerging(dim*2,name=dn));dim*=2
            self._sc.append((ns,dn))
        self.final_norm=layers.LayerNormalization(epsilon=1e-5,name="final_norm")
        self.gap=layers.GlobalAveragePooling2D(name="gap")
        self.head_drop=layers.Dropout(proj_drop,name="head_drop")
        self.head=layers.Dense(num_classes,activation="sigmoid",name="head",dtype="float32")
    def call(self,x,training=False):
        x=self.patch_embed(x)
        for ns,dn in self._sc:
            for n in ns: x=getattr(self,n)(x,training=training)
            if dn: x=getattr(self,dn)(x)
        return self.head(self.head_drop(self.gap(self.final_norm(x)),training=training))
    def get_config(self):
        return {"embed_dim":64,"depths":[2,2,6,2],"num_heads":[2,4,8,16],
                "split_sizes":[2,2,4,2],"mlp_ratio":4.0,
                "drop_path_rate":DROP_PATH_RATE,"proj_drop":DROPOUT,"num_classes":1}

CUSTOM_OBJECTS = {
    "CSWinTransformer":CSWinTransformer,"CSWinBlock":CSWinBlock,
    "CSWinAttention":CSWinAttention,"StripeAttention":StripeAttention,
    "PatchEmbedding":PatchEmbedding,"PatchMerging":PatchMerging,
    "DropPath":DropPath,"WarmupCosineDecay":WarmupCosineDecay,
}

# ══════════════════════════════════════════════════════════
# LOAD MODEL
# ══════════════════════════════════════════════════════════
print("Loading model...")
with keras.utils.custom_object_scope(CUSTOM_OBJECTS):
    model = keras.models.load_model(MODEL_PATH, compile=False)
_ = model(tf.zeros((1,)+IMG_SIZE+(3,), dtype=tf.float32), training=False)
print("Model loaded ✅")

# ══════════════════════════════════════════════════════════
# FACE DETECTOR — OpenCV DNN (ResNet SSD)
# Same accuracy as MTCNN, works on Python 3.12
# MTCNN was used in training pipeline for dataset preprocessing
# ══════════════════════════════════════════════════════════
_DNN_PROTO = "/tmp/deploy.prototxt"
_DNN_MODEL = "/tmp/res10_300x300_ssd_iter_140000.caffemodel"
_PROTO_URL = ("https://raw.githubusercontent.com/opencv/opencv/master/"
              "samples/dnn/face_detector/deploy.prototxt")
_MODEL_URL = ("https://github.com/opencv/opencv_3rdparty/raw/dnn_samples_"
              "face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel")

print("Loading face detector...")
for path, url in [(_DNN_PROTO, _PROTO_URL), (_DNN_MODEL, _MODEL_URL)]:
    if not os.path.exists(path):
        print(f"  Downloading {os.path.basename(path)}...")
        urllib.request.urlretrieve(url, path)
_dnn_net = cv2.dnn.readNetFromCaffe(_DNN_PROTO, _DNN_MODEL)
print("Face detector ready ✅")

def detect_and_crop_face(img_rgb):
    """
    Detect face using OpenCV ResNet SSD.
    Equivalent to MTCNN for inference — MTCNN used in training pipeline.
    Returns: face_norm, bbox, used_fallback
    """
    h, w      = img_rgb.shape[:2]
    blob      = cv2.dnn.blobFromImage(
        cv2.resize(img_rgb, (300, 300)), 1.0,
        (300, 300), (104.0, 177.0, 123.0))
    _dnn_net.setInput(blob)
    dets      = _dnn_net.forward()

    best_conf = 0.0
    best_box  = None
    for i in range(dets.shape[2]):
        c = float(dets[0, 0, i, 2])
        if c > best_conf:
            best_conf = c
            best_box  = dets[0, 0, i, 3:7]

    if best_box is not None and best_conf > 0.5:
        box          = best_box * np.array([w, h, w, h])
        x1, y1, x2, y2 = box.astype(int)
        x1 = max(0, x1); y1 = max(0, y1)
        x2 = min(w, x2); y2 = min(h, y2)
        face_crop    = img_rgb[y1:y2, x1:x2]
        if face_crop.size > 0:
            face_norm = cv2.resize(face_crop, IMG_SIZE).astype(np.float32) / 255.0
            bbox      = {"x": x1, "y": y1, "w": x2-x1, "h": y2-y1,
                         "conf": round(best_conf, 2)}
            return face_norm, bbox, False

    # Fallback
    print("  No face detected — using full image")
    face_norm = cv2.resize(img_rgb, IMG_SIZE).astype(np.float32) / 255.0
    return face_norm, None, True

# ══════════════════════════════════════════════════════════
# XAI COLORMAP
# ══════════════════════════════════════════════════════════
_SHAP_CMAP = LinearSegmentedColormap.from_list(
    "shap_br",
    [(0.0,"#1565C0"),(0.35,"#90CAF9"),(0.5,"#F5F5F5"),
     (0.65,"#EF9A9A"),(1.0,"#B71C1C")])

# ══════════════════════════════════════════════════════════
# XAI METHODS
# ══════════════════════════════════════════════════════════
def compute_integrated_gradients(inp, n_steps=50):
    baseline  = tf.zeros_like(inp, dtype=tf.float32)
    inp_tf    = tf.cast(inp, tf.float32)
    alphas    = tf.linspace(0.0, 1.0, n_steps)
    grads_all = []
    for alpha in alphas:
        interp = baseline + alpha * (inp_tf - baseline)
        with tf.GradientTape() as tape:
            tape.watch(interp)
            score = model(interp, training=False)[:, 0]
        grads_all.append(tape.gradient(score, interp).numpy()[0])
    avg_grads = np.mean(grads_all, axis=0)
    ig        = (inp_tf.numpy()[0] - baseline.numpy()[0]) * avg_grads
    vmax      = np.abs(ig).max() + 1e-8
    ig_signed = ig / vmax
    ig_abs    = np.abs(ig).mean(axis=-1)
    ig_abs    = gaussian_filter(ig_abs, sigma=1.5)
    if ig_abs.max() > 0: ig_abs /= ig_abs.max()
    return ig_signed, ig_abs

def compute_smoothgrad(inp, n_samples=20, noise_level=0.10):
    x_base      = tf.cast(inp, tf.float32)
    grads_accum = np.zeros_like(inp[0])
    for _ in range(n_samples):
        noise   = tf.random.normal(shape=tf.shape(x_base),
                                   stddev=noise_level, dtype=tf.float32)
        x_noisy = tf.clip_by_value(x_base + noise, 0.0, 1.0)
        with tf.GradientTape() as tape:
            tape.watch(x_noisy)
            score = model(x_noisy, training=False)[:, 0]
        grads_accum += tape.gradient(score, x_noisy).numpy()[0]
    grads_accum  /= n_samples
    vmax          = np.abs(grads_accum).max() + 1e-8
    smooth_signed = grads_accum / vmax
    smooth_abs    = np.abs(grads_accum).mean(axis=-1)
    smooth_abs    = gaussian_filter(smooth_abs, sigma=1.5)
    if smooth_abs.max() > 0: smooth_abs /= smooth_abs.max()
    return smooth_signed, smooth_abs

def compute_occlusion_map(inp):
    H, W      = IMG_SIZE
    base_prob = float(model(inp, training=False).numpy()[0, 0])
    sens_map  = np.zeros((H, W), dtype=np.float32)
    for r in range(0, H, OCC_PATCH_SIZE):
        for c in range(0, W, OCC_PATCH_SIZE):
            r2, c2 = min(H, r+OCC_PATCH_SIZE), min(W, c+OCC_PATCH_SIZE)
            patch  = inp.copy()
            patch[0, r:r2, c:c2, :] = 0.5
            sens_map[r:r2, c:c2] = abs(
                base_prob - float(model(patch, training=False).numpy()[0, 0]))
    if sens_map.max() > 0: sens_map /= sens_map.max()
    return gaussian_filter(sens_map, sigma=1.5)

def compute_region_scores(ig_abs, smooth_abs, occ_map):
    H, W     = IMG_SIZE
    combined = (ig_abs + smooth_abs + occ_map) / 3.0
    def _m(rows, cols=None):
        patch = combined[rows[0]:rows[1], :] if cols is None \
                else combined[rows[0]:rows[1], cols[0]:cols[1]]
        return float(patch.mean()) if patch.size > 0 else 0.0
    scores = {
        "Forehead":   _m([0,          int(H*.25)], [int(W*.15), int(W*.85)]),
        "Eyes/Brows": _m([int(H*.20), int(H*.45)]),
        "Nose":       _m([int(H*.40), int(H*.65)], [int(W*.30), int(W*.70)]),
        "Mouth":      _m([int(H*.60), int(H*.85)], [int(W*.20), int(W*.80)]),
        "Cheeks/Jaw": _m([int(H*.45), int(H*.90)]),
    }
    total = sum(scores.values()) + 1e-8
    return {k: v/total for k, v in scores.items()}

# ══════════════════════════════════════════════════════════
# BUILD XAI FIGURE
# ══════════════════════════════════════════════════════════
def build_xai_figure(face_norm, orig_rgb, bbox, label, conf, prob,
                     ig_signed, ig_abs, smooth_signed, smooth_abs,
                     occ_map, region_scores, used_fallback):

    ig_mean    = ig_signed.mean(axis=-1)
    ig_pos     = np.maximum( ig_mean, 0)
    ig_neg     = np.maximum(-ig_mean, 0)
    sm_mean    = smooth_signed.mean(axis=-1)
    combined   = (ig_abs + smooth_abs + occ_map) / 3.0
    top_region = max(region_scores, key=region_scores.get)
    edge       = "#2ecc71" if label=="REAL" else "#e74c3c"
    bg         = "#2ecc71" if label=="REAL" else "#e74c3c"

    fig = plt.figure(figsize=(22, 16))
    gs  = gridspec.GridSpec(3, 4, figure=fig,
                            hspace=0.42, wspace=0.30,
                            top=0.92, bottom=0.04)

    # ── ROW 1: Prediction ──
    ax = fig.add_subplot(gs[0, 0])
    ax.imshow(orig_rgb); ax.axis("off")
    ax.set_title("Input Image", fontsize=11)
    if bbox:
        rect = patches.Rectangle(
            (bbox["x"], bbox["y"]), bbox["w"], bbox["h"],
            linewidth=3, edgecolor=edge, facecolor="none")
        ax.add_patch(rect)
        ax.text(bbox["x"], max(0, bbox["y"]-8),
                f"Face {bbox['conf']:.2f}",
                color=edge, fontsize=9, fontweight="bold",
                bbox=dict(facecolor="black", alpha=0.55, pad=2))

    ax = fig.add_subplot(gs[0, 1])
    ax.imshow(face_norm); ax.axis("off")
    ax.set_title("Extracted Face" if not used_fallback
                 else "Full Image (no face)", fontsize=11)

    ax = fig.add_subplot(gs[0, 2:])
    ax.set_facecolor(bg); ax.axis("off")
    ax.text(0.5, 0.64, label,
            ha="center", va="center", fontsize=58,
            fontweight="bold", color="white", transform=ax.transAxes)
    ax.text(0.5, 0.40, f"Confidence: {conf*100:.1f}%",
            ha="center", va="center", fontsize=24,
            color="white", transform=ax.transAxes)
    ax.text(0.5, 0.24,
            f"REAL: {prob*100:.1f}%   |   FAKE: {(1-prob)*100:.1f}%",
            ha="center", va="center", fontsize=14,
            color="white", alpha=0.9, transform=ax.transAxes)
    ax.text(0.5, 0.10,
            f"Score: {prob:.4f}   |   Threshold: {THRESHOLD}",
            ha="center", va="center", fontsize=11,
            color="white", alpha=0.75, transform=ax.transAxes)
    if used_fallback:
        ax.text(0.5, 0.02, "No face detected — full image used",
                ha="center", va="center", fontsize=9,
                color="yellow", transform=ax.transAxes)
    ax.set_title("Prediction", fontsize=11)

    # ── ROW 2: Attribution maps ──
    def _heat(parent_gs, data, cmap, title, alpha=0.65, symmetric=False):
        ax = fig.add_subplot(parent_gs)
        ax.imshow(face_norm, alpha=0.45)
        vmax_ = max(abs(data).max(), 1e-8)
        kw = dict(cmap=cmap, alpha=alpha)
        if symmetric: kw.update(vmin=-vmax_, vmax=vmax_)
        else:         kw.update(vmin=0, vmax=1)
        im = ax.imshow(data, **kw)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        ax.set_title(title, fontsize=10); ax.axis("off")

    _heat(gs[1,0], ig_mean, _SHAP_CMAP,
          "Integrated Gradients\n(Red→REAL, Blue→FAKE)", symmetric=True)
    _heat(gs[1,1], ig_abs,  "hot",
          "IG Absolute Saliency\n(Hot = high importance)")
    _heat(gs[1,2], sm_mean, _SHAP_CMAP,
          f"SmoothGrad (n={SHAP_N_BG})\n(Red→REAL, Blue→FAKE)", symmetric=True)
    _heat(gs[1,3], occ_map, "YlOrRd",
          f"Occlusion Sensitivity\n(patch={OCC_PATCH_SIZE}px)")

    # ── ROW 3: Decomposition + region bar ──
    ax = fig.add_subplot(gs[2, 0])
    ax.imshow(face_norm, alpha=0.4)
    im = ax.imshow(ig_pos, cmap="Reds", alpha=0.8, vmin=0, vmax=1)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title("IG Positive Contribution\n(Evidence FOR real)", fontsize=10)
    ax.axis("off")

    ax = fig.add_subplot(gs[2, 1])
    ax.imshow(face_norm, alpha=0.4)
    im = ax.imshow(ig_neg, cmap="Blues", alpha=0.8, vmin=0, vmax=1)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title("IG Negative Contribution\n(Evidence FOR fake)", fontsize=10)
    ax.axis("off")

    ax = fig.add_subplot(gs[2, 2])
    ax.imshow(face_norm, alpha=0.35)
    im = ax.imshow(combined, cmap="magma", alpha=0.75, vmin=0, vmax=1)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    flat_idx = np.argsort(combined.ravel())[::-1][:5]
    hy, hx   = np.unravel_index(flat_idx, combined.shape)
    ax.scatter(hx, hy, s=60, c="yellow", marker="*",
               edgecolors="black", linewidths=0.5, zorder=5)
    ax.set_title("Combined Attribution Map\n(★ = top-5 critical pixels)",
                 fontsize=10)
    ax.axis("off")

    ax = fig.add_subplot(gs[2, 3])
    regions    = list(region_scores.keys())
    scores_pct = [region_scores[r]*100 for r in regions]
    bar_cols   = ["#E53935","#FB8C00","#43A047","#1E88E5","#8E24AA"]
    bars       = ax.barh(regions, scores_pct, color=bar_cols,
                         edgecolor="white", linewidth=1.2)
    for bar, val in zip(bars, scores_pct):
        ax.text(val+0.3, bar.get_y()+bar.get_height()/2,
                f"{val:.1f}%", va="center", fontsize=10, fontweight="bold")
    ax.set_xlim(0, max(scores_pct)*1.35)
    ax.set_xlabel("Relative Importance (%)", fontsize=10)
    ax.set_title("Face Region Importance\n(IG + SmoothGrad + Occlusion)",
                 fontsize=10)
    ax.spines[["top","right"]].set_visible(False)
    ax.text(0.97, 0.04, f"Key: {top_region}",
            transform=ax.transAxes, ha="right",
            fontsize=9, style="italic", color="dimgray")

    verdict_col = "#1a7a3c" if label=="REAL" else "#c0392b"
    fig.suptitle(
        f"CSWin Transformer — DeepFake Detection Explanation\n"
        f"Verdict: {label}  ({conf*100:.1f}% confident)   "
        f"Score={prob:.4f}   Key region: {top_region}",
        fontsize=13, fontweight="bold", color=verdict_col, y=0.97)

    return fig, top_region

# ══════════════════════════════════════════════════════════
# GRADIO PREDICT FUNCTION
# ══════════════════════════════════════════════════════════
def predict_and_explain(pil_image):
    if pil_image is None:
        return {"REAL ✅": 0.0, "FAKE ⚠️": 0.0}, None

    orig_rgb = np.array(pil_image.convert("RGB"))

    # Face detection
    face_norm, bbox, used_fallback = detect_and_crop_face(orig_rgb)
    inp = np.expand_dims(face_norm, axis=0)

    # Predict
    prob      = float(model(inp, training=False).numpy()[0, 0])
    real_prob = round(prob, 4)
    fake_prob = round(1.0 - prob, 4)
    label     = "REAL" if real_prob >= THRESHOLD else "FAKE"
    conf      = real_prob if label == "REAL" else fake_prob

    print(f"\n  Verdict: {label}  ({conf*100:.1f}%)  [score={prob:.4f}]")
    print(f"  Face detected: {bbox is not None}")

    # XAI
    print("  [1/3] Integrated Gradients...")
    ig_signed, ig_abs = compute_integrated_gradients(inp, n_steps=50)

    print("  [2/3] SmoothGrad...")
    smooth_signed, smooth_abs = compute_smoothgrad(inp, n_samples=SHAP_N_BG)

    print("  [3/3] Occlusion sensitivity...")
    occ_map = compute_occlusion_map(inp)

    region_scores = compute_region_scores(ig_abs, smooth_abs, occ_map)

    fig, top_region = build_xai_figure(
        face_norm=face_norm, orig_rgb=orig_rgb, bbox=bbox,
        label=label, conf=conf, prob=prob,
        ig_signed=ig_signed, ig_abs=ig_abs,
        smooth_signed=smooth_signed, smooth_abs=smooth_abs,
        occ_map=occ_map, region_scores=region_scores,
        used_fallback=used_fallback
    )

    out_path = os.path.join(OUT_DIR, "xai_output.png")
    fig.savefig(out_path, dpi=130, bbox_inches="tight", facecolor="white")
    plt.close()

    print(f"  Top region: {top_region} ({region_scores[top_region]*100:.1f}%)")

    return {"REAL ✅": real_prob, "FAKE ⚠️": fake_prob}, out_path

# ══════════════════════════════════════════════════════════
# FRPC FIX
# ══════════════════════════════════════════════════════════
frpc_dir = "/root/.cache/huggingface/gradio/frpc"
os.makedirs(frpc_dir, exist_ok=True)
dest = os.path.join(frpc_dir, "frpc_linux_amd64_v0.3")
if not os.path.exists(dest):
    try:
        print("Downloading frpc...")
        urllib.request.urlretrieve(
            "https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_linux_amd64",
            dest)
        os.chmod(dest, 0o755)
        print("frpc ready ✅")
    except Exception as e:
        print(f"frpc failed: {e}")

# ══════════════════════════════════════════════════════════
# GRADIO UI
# ══════════════════════════════════════════════════════════
demo = gr.Interface(
    fn=predict_and_explain,
    inputs=gr.Image(type="pil", label="Upload Face Image"),
    outputs=[
        gr.Label(num_top_classes=2, label="Prediction"),
        gr.Image(label="XAI Explanation — IG + SmoothGrad + Occlusion")
    ],
    title="🛡️ DeepShield — Deepfake Detection System",
    description=(
        "Upload a face image to detect if it's Real or Fake.\n"
        "Face detection using OpenCV ResNet SSD "
        "(MTCNN used in training pipeline).\n"
        "Output 1: Real/Fake probability scores\n"
        "Output 2: Full XAI — Integrated Gradients + SmoothGrad + "
        "Occlusion Sensitivity + Region Importance\n"
        "Model: CSWin Transformer | SVCE 2026"
    ),
    allow_flagging="never",
    theme=gr.themes.Soft()
)

demo.queue(
    max_size=5,
    default_concurrency_limit=1
).launch(
    share=True,
    server_name="0.0.0.0",
    server_port=7860,
    show_error=True,
    favicon_path=None
)

2026-05-12 15:08:42.712338: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778598523.025350      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778598523.120243      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778598523.943098      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778598523.943151      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778598523.943155      16 computation_placer.cc:177] computation placer alr

Loading model...


2026-05-12 15:09:20.061794: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model loaded ✅
Loading face detector...
Face detector ready ✅
frpc ready ✅
* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://dd47176847d366ae27.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
